In [1]:
import os
import glob
import pandas as pd

# Define the base directory where result files are stored
base_dir = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\2.source_code\Step5_Geo_RF_trial\Food_Crisis_Cluster\results'
rf_pattern = os.path.join(base_dir, 'results_df_gp_fs*_*_*.csv')
rf_files = glob.glob(rf_pattern)
scope_to_lag = {1: 4, 2: 8, 3: 12}

# Initialize data storage
probit_data = {}
fewsnet_data = {}
rf_data = {}
xgboost_data = {}

# Group RF files by forecasting scope and combine across year ranges
rf_by_scope = {}
for file_path in rf_files:
    filename = os.path.basename(file_path)
    try:
        # Extract scope from: results_df_gp_fs1_2015_2016.csv -> scope=1
        parts = filename.split('_fs')[1].split('_')
        scope = int(parts[0])
        start_year = int(parts[1])
        end_year = int(parts[2].split('.')[0])
        lag_months = scope_to_lag[scope]
        
        df = pd.read_csv(file_path)
        print(f"  Found GeoRF results for scope {scope} ({lag_months}-month), years {start_year}-{end_year}: {filename}")
        
        if lag_months not in rf_by_scope:
            rf_by_scope[lag_months] = []
        rf_by_scope[lag_months].append(df)
        
    except (ValueError, IndexError, KeyError) as e:
        print(f"  Warning: Could not parse filename {filename}: {e}")

# Combine RF data across year ranges for each scope
for lag_months, dfs in rf_by_scope.items():
    combined_df = pd.concat(dfs, ignore_index=True)
    
    # Create year-quarter identifier
    if 'quarter' in combined_df.columns:
        combined_df['year_quarter'] = combined_df['year'].astype(str) + '-Q' + combined_df['quarter'].astype(str)
    else:
        combined_df['year_quarter'] = combined_df['year'].astype(str)
    
    combined_df = combined_df.sort_values(['year'] + (['quarter'] if 'quarter' in combined_df.columns else [])).reset_index(drop=True)
    rf_data[lag_months] = combined_df
    print(f"  Combined {len(dfs)} files for {lag_months}-month lag: {len(combined_df)} total records")

  Found GeoRF results for scope 1 (4-month), years 2013-2013: results_df_gp_fs1_2013_2013.csv
  Found GeoRF results for scope 1 (4-month), years 2014-2014: results_df_gp_fs1_2014_2014.csv
  Found GeoRF results for scope 1 (4-month), years 2015-2015: results_df_gp_fs1_2015_2015.csv
  Found GeoRF results for scope 1 (4-month), years 2016-2016: results_df_gp_fs1_2016_2016.csv
  Found GeoRF results for scope 1 (4-month), years 2017-2017: results_df_gp_fs1_2017_2017.csv
  Found GeoRF results for scope 1 (4-month), years 2018-2018: results_df_gp_fs1_2018_2018.csv
  Found GeoRF results for scope 1 (4-month), years 2019-2019: results_df_gp_fs1_2019_2019.csv
  Found GeoRF results for scope 1 (4-month), years 2020-2020: results_df_gp_fs1_2020_2020.csv
  Found GeoRF results for scope 1 (4-month), years 2021-2021: results_df_gp_fs1_2021_2021.csv
  Found GeoRF results for scope 1 (4-month), years 2022-2022: results_df_gp_fs1_2022_2022.csv
  Found GeoRF results for scope 1 (4-month), years 2023-2023

In [2]:
# for rf_data, concate all dfs into one df
# for each scope add a column 'lag_months' to indicate the scope
for lag_months, df in rf_data.items():
    df['lag_months'] = lag_months


all_rf_data = pd.concat(rf_data.values(), ignore_index=True)
# mean of precision, recall, f1_score for each lag_months
rf_summary = all_rf_data.groupby('lag_months').agg({
    'precision(1)': 'mean',
    'recall(1)': 'mean',
    'f1(1)': 'mean',
    'precision_base(1)': 'mean',
    'recall_base(1)': 'mean',
    'f1_base(1)': 'mean'
}).reset_index()

In [3]:
all_rf_data['better_than_base'] = all_rf_data['f1(1)'] >= all_rf_data['f1_base(1)']

#mean of better_than_base for each lag_months
rf_better_summary = all_rf_data.groupby('lag_months').agg({
    'better_than_base': 'mean'
}).reset_index()

#save rf_better_summary to csv
rf_better_summary.to_csv(os.path.join(base_dir, 'rf_better_summary.csv'), index=False)

In [4]:
print("\nSearching for XGBoost results...")
xgb_pattern = os.path.join(base_dir, 'results_df_xgb_gp_fs*_*_*.csv')
xgb_files = glob.glob(xgb_pattern)

# Group XGB files by forecasting scope and combine across year ranges
xgb_by_scope = {}
for file_path in xgb_files:
    filename = os.path.basename(file_path)
    try:
        # Extract scope from: results_df_xgb_gp_fs1_2015_2015.csv or results_df_xgb_gp_fs1_2015_2016.csv -> scope=1
        parts = filename.split('_fs')[1].split('_')
        scope = int(parts[0])
        start_year = int(parts[1])
        end_year = int(parts[2].split('.')[0])
        lag_months = scope_to_lag[scope]
        
        df = pd.read_csv(file_path)
        print(f"  Found XGBoost results for scope {scope} ({lag_months}-month), years {start_year}-{end_year}: {filename}")
        
        if lag_months not in xgb_by_scope:
            xgb_by_scope[lag_months] = []
        xgb_by_scope[lag_months].append(df)
        
    except (ValueError, IndexError, KeyError) as e:
        print(f"  Warning: Could not parse filename {filename}: {e}")

# Combine XGBoost data across year ranges for each scope
for lag_months, dfs in xgb_by_scope.items():
    combined_df = pd.concat(dfs, ignore_index=True)
    
    # Create year-quarter identifier
    if 'quarter' in combined_df.columns:
        combined_df['year_quarter'] = combined_df['year'].astype(str) + '-Q' + combined_df['quarter'].astype(str)
    else:
        combined_df['year_quarter'] = combined_df['year'].astype(str)
        
    combined_df = combined_df.sort_values(['year'] + (['quarter'] if 'quarter' in combined_df.columns else [])).reset_index(drop=True)
    xgboost_data[lag_months] = combined_df
    print(f"  Combined {len(dfs)} files for {lag_months}-month lag: {len(combined_df)} total records")


Searching for XGBoost results...
  Found XGBoost results for scope 1 (4-month), years 2013-2013: results_df_xgb_gp_fs1_2013_2013.csv
  Found XGBoost results for scope 1 (4-month), years 2014-2014: results_df_xgb_gp_fs1_2014_2014.csv
  Found XGBoost results for scope 1 (4-month), years 2015-2015: results_df_xgb_gp_fs1_2015_2015.csv
  Found XGBoost results for scope 1 (4-month), years 2016-2016: results_df_xgb_gp_fs1_2016_2016.csv
  Found XGBoost results for scope 1 (4-month), years 2017-2017: results_df_xgb_gp_fs1_2017_2017.csv
  Found XGBoost results for scope 1 (4-month), years 2018-2018: results_df_xgb_gp_fs1_2018_2018.csv
  Found XGBoost results for scope 1 (4-month), years 2019-2019: results_df_xgb_gp_fs1_2019_2019.csv
  Found XGBoost results for scope 1 (4-month), years 2020-2020: results_df_xgb_gp_fs1_2020_2020.csv
  Found XGBoost results for scope 1 (4-month), years 2021-2021: results_df_xgb_gp_fs1_2021_2021.csv
  Found XGBoost results for scope 1 (4-month), years 2022-2022: re

In [5]:
# for xgboost_data, concate all dfs into one df
for lag_months, df in xgboost_data.items():
    df['lag_months'] = lag_months
    
all_xgb_data = pd.concat(xgboost_data.values(), ignore_index=True)
# mean of precision, recall, f1_score for each lag_months
xgb_summary = all_xgb_data.groupby('lag_months').agg({
    'precision(1)': 'mean',
    'recall(1)': 'mean',
    'f1(1)': 'mean',
    'precision_base(1)': 'mean',
    'recall_base(1)': 'mean',
    'f1_base(1)': 'mean'
}).reset_index()

In [6]:
#xgb_better_summary
all_xgb_data['better_than_base'] = all_xgb_data['f1(1)'] >= all_xgb_data['f1_base(1)']
#mean of better_than_base for each lag_months
xgb_better_summary = all_xgb_data.groupby('lag_months').agg({
    'better_than_base': 'mean'
}).reset_index()

#save xgb_better_summary to csv
xgb_better_summary.to_csv(os.path.join(base_dir, 'xgb_better_summary.csv'), index=False)

In [7]:
#export rf_summary and xgb_summary to csv
rf_summary.to_csv(os.path.join(base_dir, 'rf_summary.csv'), index=False)
xgb_summary.to_csv(os.path.join(base_dir, 'xgb_summary.csv'), index=False)